# Image generation — The Viridian Archive

Select **Runtime → Change runtime type → T4 GPU**, then run the training cell. This notebook trains a small unconditional DDPM on real animal faces. Four controlled runs vary data size (500/1000), resolution (32/64), and epochs (5/20). These are deliberately small lo-fi experiments, not a guarantee of coherent hybrids. Creature names are interpretations assigned after generation, not conditioning prompts.

Dataset: [huggan/AFHQ](https://huggingface.co/datasets/huggan/AFHQ). Method: [Diffusers DDPM tutorial](https://huggingface.co/docs/diffusers/tutorials/basic_training). Original AFHQ: [StarGAN v2](https://github.com/clovaai/stargan-v2). Check the original dataset terms before reuse; this project is an educational experiment.

Checkpoints survive reruns in the **same runtime**. Download results before disconnecting; `/content` is temporary. A new runtime starts over. No fabricated metrics are supplied.


In [ ]:
# Postcards from my jungle — image experiments
# Run on Colab with a T4 GPU. Based on the Hugging Face Diffusers DDPM tutorial.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.1', 'datasets==4.1.1', 'accelerate==1.10.1', 'transformers==4.56.2', 'soundfile', 'sentencepiece'])
import os, json, random, time, gc, platform
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from diffusers import UNet2DModel, DDPMScheduler, DDPMPipeline
from PIL import Image, ImageOps, ImageDraw
from tqdm.auto import tqdm
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
ROOT = Path('/content/postcards-from-my-jungle'); ROOT.mkdir(exist_ok=True)
(ROOT/'images').mkdir(exist_ok=True)
RUNS = ROOT/'experiments'; RUNS.mkdir(exist_ok=True)
SEED=498
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0), flush=True)
raw=load_dataset('huggan/AFHQ', split='train').shuffle(seed=SEED)
# The larger sets include the smaller sets; the holdout is always disjoint.
train_raw=raw.select(range(1000)); valid_raw=raw.select(range(1000,1100))
print('Dataset ready:', len(train_raw), 'training and', len(valid_raw), 'held-out images', flush=True)
class Animals(Dataset):
    def __init__(self, ds, size, augment):
        self.ds=ds
        self.tx=transforms.Compose([transforms.Resize((size,size)), *([transforms.RandomHorizontalFlip()] if augment else []), transforms.ToTensor(), transforms.Normalize([.5]*3,[.5]*3)])
    def __len__(self): return len(self.ds)
    def __getitem__(self,i): return self.tx(self.ds[i]['image'].convert('RGB'))
experiments=[
    dict(name='small-32', count=500, resolution=32, epochs=5),
    dict(name='more-data-32', count=1000, resolution=32, epochs=5),
    dict(name='higher-resolution-64', count=1000, resolution=64, epochs=5),
    dict(name='longer-training-64', count=1000, resolution=64, epochs=20),
]
all_results=[]
for config in experiments:
    folder=RUNS/config['name']; folder.mkdir(exist_ok=True)
    torch.manual_seed(SEED); random.seed(SEED)
    model=UNet2DModel(sample_size=config['resolution'], in_channels=3,out_channels=3,layers_per_block=1,block_out_channels=(32,64,128),down_block_types=('DownBlock2D','DownBlock2D','DownBlock2D'),up_block_types=('UpBlock2D','UpBlock2D','UpBlock2D'),norm_num_groups=8).cuda()
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)
    noise_scheduler=DDPMScheduler(num_train_timesteps=1000,beta_schedule='squaredcos_cap_v2')
    scaler=torch.amp.GradScaler('cuda')
    loader=DataLoader(Animals(train_raw.select(range(config['count'])),config['resolution'],True),batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
    valid=DataLoader(Animals(valid_raw,config['resolution'],False),batch_size=25)
    start=0; history=[]; checkpoint=folder/'checkpoint.pt'
    if checkpoint.exists():
        state=torch.load(checkpoint,map_location='cuda',weights_only=False)
        model.load_state_dict(state['model']); optimizer.load_state_dict(state['optimizer']); scaler.load_state_dict(state['scaler']); start=state['epoch']; history=state['history']
    for epoch in range(start,config['epochs']):
        model.train(); total=0; seen=0; began=time.time()
        for clean in tqdm(loader,desc=f"{config['name']} epoch {epoch+1}/{config['epochs']}",leave=False):
            clean=clean.cuda(); noise=torch.randn_like(clean); steps=torch.randint(0,1000,(clean.shape[0],),device='cuda')
            noisy=noise_scheduler.add_noise(clean,noise,steps)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16): loss=F.mse_loss(model(noisy,steps).sample.float(),noise.float())
            scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer); scaler.update(); total+=loss.item()*len(clean);seen+=len(clean)
        model.eval(); val_loss=0; vn=0; vg=torch.Generator(device='cuda').manual_seed(42)
        with torch.no_grad():
            for clean in valid:
                clean=clean.cuda(); noise=torch.randn(clean.shape,device='cuda',generator=vg); steps=torch.randint(0,1000,(len(clean),),device='cuda',generator=vg)
                with torch.autocast('cuda',dtype=torch.float16): loss=F.mse_loss(model(noise_scheduler.add_noise(clean,noise,steps),steps).sample.float(),noise.float())
                val_loss+=loss.item()*len(clean); vn+=len(clean)
        row=dict(epoch=epoch+1,training_loss=total/seen,validation_loss=val_loss/vn,seconds=time.time()-began);history.append(row)
        print(config['name'],row,flush=True)
        torch.save(dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),epoch=epoch+1,history=history),checkpoint)
        (folder/'metrics.json').write_text(json.dumps(dict(config=config,history=history),indent=2))
    pipe=DDPMPipeline(unet=model,scheduler=noise_scheduler).to('cuda');pipe.set_progress_bar_config(disable=True)
    previews=pipe(batch_size=4,num_inference_steps=100,generator=torch.Generator(device='cuda').manual_seed(700)).images
    sheet=Image.new('RGB',(config['resolution']*4,config['resolution']))
    for i,img in enumerate(previews):sheet.paste(img,(i*config['resolution'],0))
    sheet.resize((1024,256),Image.Resampling.NEAREST).save(folder/'comparison.png')
    pipe.save_pretrained(folder/'model')
    all_results.append(dict(config=config,last=history[-1]))
    (RUNS/'summary.json').write_text(json.dumps(all_results,indent=2))
    del pipe,model,optimizer,scaler,loader,valid;gc.collect();torch.cuda.empty_cache()
# Automatic default: the longest run. Review all four comparison sheets and change
# SELECTED_RUN if a different run looks better. Loss alone is not visual quality.
SELECTED_RUN='longer-training-64'
pipe=DDPMPipeline.from_pretrained(RUNS/SELECTED_RUN/'model').to('cuda');pipe.set_progress_bar_config(disable=True)
manifest=[]
for start in tqdm(range(0,50,5),desc='Generate fifty images'):
    batch=pipe(batch_size=5,num_inference_steps=250,generator=[torch.Generator(device='cuda').manual_seed(10000+i) for i in range(start,start+5)]).images
    for i,img in enumerate(batch,start):
        filename=f'animal-{i+1:03d}.png'; img.save(ROOT/'images'/filename)
        manifest.append(dict(src='images/'+filename,specimen=i%16+1,kind='trained',seed=10000+i,run=SELECTED_RUN))
(ROOT/'image-manifest.json').write_text(json.dumps(manifest,indent=2))
(ROOT/'image-run.json').write_text(json.dumps(dict(status='completed',dataset='huggan/AFHQ',dataset_fingerprint=raw._fingerprint,seed=SEED,selected_run=SELECTED_RUN,experiments=all_results,torch=torch.__version__,gpu=torch.cuda.get_device_name(0),count=50),indent=2))
contact=Image.new('RGB',(640,320))
for i,item in enumerate(manifest):
    tile=Image.open(ROOT/item['src']).resize((64,64),Image.Resampling.NEAREST);contact.paste(tile,((i%10)*64,(i//10)*64))
contact.save(RUNS/'fifty-images.png');display(contact.resize((1280,640),Image.Resampling.NEAREST))
del pipe;gc.collect();torch.cuda.empty_cache()
print('IMAGE GENERATION COMPLETE: 50 PNGs and four experiment records.',flush=True)



## Inspect and compare

Open the four `experiments/*/comparison.png` sheets. The fixed seeds make the comparison more useful, but different resolutions prevent identical image trajectories. Training and held-out noise prediction losses are diagnostics, not perceptual quality scores. Describe anatomy, diversity, memorization, and recognizability; change SELECTED_RUN and rerun the generation section if appropriate.


In [ ]:
import shutil
from google.colab import files
# Exclude large model checkpoints from the website asset download.
stage=Path('/content/image-export'); stage.mkdir(exist_ok=True)
shutil.copytree(ROOT/'images',stage/'images',dirs_exist_ok=True)
for name in ['image-manifest.json','image-run.json']: shutil.copy2(ROOT/name,stage/name)
shutil.copytree(RUNS,stage/'experiments',dirs_exist_ok=True,ignore=shutil.ignore_patterns('model','checkpoint.pt'))
shutil.make_archive('/content/jungle-images','zip',stage)
files.download('/content/jungle-images.zip')
